# Notebook 3: Fine-Tuning SAM 3

**Master's Thesis: Thai Text-to-Segment System**

This notebook demonstrates how to fine-tune SAM 3 (Segment Anything Model 3) on Thai text segmentation data.

## Overview
1. Prepare Dataset - Load and preprocess training data
2. Set Up Fine-Tuning - Configure LoRA and hyperparameters
3. Training Loop - Train with logging and checkpointing
4. Evaluation - Compare before/after performance

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision torchaudio
# !pip install transformers accelerate
# !pip install peft  # For LoRA
# !pip install wandb  # For experiment tracking
# !pip install opencv-python-headless
# !pip install albumentations
# !pip install matplotlib seaborn
# !pip install scikit-learn
# !pip install tqdm

In [ ]:
import os
import sys
import json
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import torchvision.transforms as T
from transformers import AutoModel, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Union
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Configuration Class

In [ ]:
@dataclass
class TrainingConfig:
    """Configuration for SAM 3 fine-tuning"""
    # Model
    model_name: str = "facebook/sam-vit-base"  # or sam-vit-huge, sam-vit-large
    
    # Data paths
    data_dir: str = "./data/thai_text_segmentation"
    train_json: str = "train_annotations.json"
    val_json: str = "val_annotations.json"
    output_dir: str = "./outputs/sam3_finetuned"
    checkpoint_dir: str = "./checkpoints"
    
    # Training hyperparameters
    batch_size: int = 4
    num_epochs: int = 20
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    warmup_steps: int = 100
    max_grad_norm: float = 1.0
    
    # Image settings
    image_size: int = 1024
    mask_size: int = 256
    
    # LoRA settings
    use_lora: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = None
    
    # Logging
    log_interval: int = 10
    save_interval: int = 5
    eval_interval: int = 1
    
    # Augmentation
    use_augmentation: bool = True
    
    def __post_init__(self):
        if self.lora_target_modules is None:
            self.lora_target_modules = ["q_proj", "v_proj", "k_proj", "out_proj"]
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.checkpoint_dir, exist_ok=True)

config = TrainingConfig()
print(f"Configuration:")
for key, value in vars(config).items():
    print(f"  {key}: {value}")

## 3. Prepare Dataset

### 3.1 Create/Load Dataset

In [ ]:
class ThaiTextSegmentationDataset(Dataset):
    """
    Dataset for Thai text segmentation.
    Supports polygon annotations for text regions.
    """
    
    def __init__(
        self,
        annotations_file: str,
        image_dir: str,
        processor,
        config: TrainingConfig,
        transform=None,
        is_training: bool = True
    ):
        self.image_dir = image_dir
        self.processor = processor
        self.config = config
        self.transform = transform
        self.is_training = is_training
        
        # Load annotations
        with open(annotations_file, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)
        
        # Convert to list format if needed
        if isinstance(self.annotations, dict) and 'images' in self.annotations:
            # COCO format
            self.data = self._parse_coco_format()
        else:
            self.data = self.annotations
        
        print(f"Loaded {len(self.data)} samples from {annotations_file}")
    
    def _parse_coco_format(self):
        """Parse COCO format annotations"""
        images = {img['id']: img for img in self.annotations['images']}
        annotations = self.annotations['annotations']
        
        # Group annotations by image
        data = []
        for img_id, img_info in images.items():
            img_annotations = [ann for ann in annotations if ann['image_id'] == img_id]
            if img_annotations:
                data.append({
                    'image_path': os.path.join(self.image_dir, img_info['file_name']),
                    'annotations': img_annotations,
                    'width': img_info['width'],
                    'height': img_info['height']
                })
        return data
    
    def __len__(self):
        return len(self.data)
    
    def _create_mask(self, annotations, height, width):
        """Create binary mask from polygon annotations"""
        mask = np.zeros((height, width), dtype=np.uint8)
        
        for ann in annotations:
            if 'segmentation' in ann:
                # COCO polygon format
                if isinstance(ann['segmentation'], list):
                    for polygon in ann['segmentation']:
                        pts = np.array(polygon).reshape(-1, 2).astype(np.int32)
                        cv2.fillPoly(mask, [pts], 1)
                # RLE format
                elif isinstance(ann['segmentation'], dict):
                    from pycocotools import mask as maskUtils
                    rle = ann['segmentation']
                    m = maskUtils.decode(rle)
                    mask = np.maximum(mask, m)
        
        return mask
    
    def _get_bounding_boxes(self, annotations):
        """Extract bounding boxes from annotations"""
        boxes = []
        for ann in annotations:
            if 'bbox' in ann:
                # COCO bbox format: [x, y, width, height]
                x, y, w, h = ann['bbox']
                boxes.append([x, y, x + w, y + h])
        return boxes
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load image
        image_path = item['image_path']
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        original_height, original_width = image.shape[:2]
        
        # Create mask
        mask = self._create_mask(
            item['annotations'],
            original_height,
            original_width
        )
        
        # Get bounding boxes
        boxes = self._get_bounding_boxes(item['annotations'])
        
        # Apply augmentations
        if self.transform and self.is_training:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        else:
            # Basic resize and normalize
            image = cv2.resize(image, (self.config.image_size, self.config.image_size))
            mask = cv2.resize(mask, (self.config.mask_size, self.config.mask_size),
                            interpolation=cv2.INTER_NEAREST)
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        # Resize mask to model output size
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask).float()
        
        # Prepare prompt (use center point of first bbox as default)
        input_points = None
        if boxes:
            x1, y1, x2, y2 = boxes[0]
            center_x = (x1 + x2) / 2 / original_width * self.config.image_size
            center_y = (y1 + y2) / 2 / original_height * self.config.image_size
            input_points = [[[center_x, center_y]]]
        
        return {
            'image': image,
            'mask': mask,
            'input_points': input_points,
            'image_path': image_path,
            'original_size': (original_height, original_width)
        }

### 3.2 Data Augmentations

In [ ]:
def get_training_transforms(config: TrainingConfig):
    """Get training data augmentation pipeline"""
    return A.Compose([
        # Geometric transformations
        A.RandomResizedCrop(
            size=(config.image_size, config.image_size),
            scale=(0.8, 1.0),
            ratio=(0.9, 1.1),
            p=0.5
        ),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.1,
            rotate_limit=15,
            p=0.5
        ),
        
        # Color augmentations
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=1.0
            ),
            A.RandomGamma(gamma_limit=(80, 120), p=1.0),
        ], p=0.5),
        
        # Noise and blur
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.3),
        
        # Normalize
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ])

def get_validation_transforms(config: TrainingConfig):
    """Get validation data transformation pipeline"""
    return A.Compose([
        A.Resize(config.image_size, config.image_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ])

# Test augmentations
print("Training transforms:")
print(get_training_transforms(config))
print("\nValidation transforms:")
print(get_validation_transforms(config))

### 3.3 Create Data Loaders

In [ ]:
def create_dataloaders(config: TrainingConfig, processor):
    """Create training and validation data loaders"""
    
    # Check if annotation files exist, create dummy data if not
    train_json_path = os.path.join(config.data_dir, config.train_json)
    val_json_path = os.path.join(config.data_dir, config.val_json)
    
    if not os.path.exists(train_json_path):
        print(f"Warning: {train_json_path} not found. Creating dummy dataset...")
        os.makedirs(config.data_dir, exist_ok=True)
        create_dummy_dataset(config)
    
    # Create datasets
    train_transform = get_training_transforms(config) if config.use_augmentation else get_validation_transforms(config)
    val_transform = get_validation_transforms(config)
    
    train_dataset = ThaiTextSegmentationDataset(
        annotations_file=train_json_path,
        image_dir=os.path.join(config.data_dir, 'images'),
        processor=processor,
        config=config,
        transform=train_transform,
        is_training=True
    )
    
    val_dataset = ThaiTextSegmentationDataset(
        annotations_file=val_json_path,
        image_dir=os.path.join(config.data_dir, 'images'),
        processor=processor,
        config=config,
        transform=val_transform,
        is_training=False
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    return train_loader, val_loader

def create_dummy_dataset(config: TrainingConfig, num_samples=100):
    """Create dummy dataset for demonstration"""
    print("Creating dummy dataset...")
    
    images_dir = os.path.join(config.data_dir, 'images')
    os.makedirs(images_dir, exist_ok=True)
    
    train_annotations = []
    val_annotations = []
    
    for i in range(num_samples):
        # Create dummy image with text-like regions
        img = np.ones((512, 512, 3), dtype=np.uint8) * 255
        
        # Add some rectangles (simulating text) 
        num_text_regions = random.randint(3, 8)
        annotations = []
        
        for j in range(num_text_regions):
            x = random.randint(50, 400)
            y = random.randint(50, 400)
            w = random.randint(50, 150)
            h = random.randint(20, 60)
            
            # Draw rectangle
            color = (random.randint(0, 100), random.randint(0, 100), random.randint(0, 100))
            cv2.rectangle(img, (x, y), (x + w, y + h), color, -1)
            
            # Create polygon annotation
            polygon = [
                x, y,
                x + w, y,
                x + w, y + h,
                x, y + h
            ]
            
            annotations.append({
                'id': j,
                'image_id': i,
                'category_id': 1,
                'bbox': [x, y, w, h],
                'segmentation': [polygon],
                'area': w * h,
                'iscrowd': 0
            })
        
        # Save image
        img_path = os.path.join(images_dir, f'image_{i:04d}.jpg')
        cv2.imwrite(img_path, img)
        
        # Add to annotations
        image_ann = {
            'image_id': i,
            'file_name': f'image_{i:04d}.jpg',
            'width': 512,
            'height': 512,
            'annotations': annotations
        }
        
        if i < int(num_samples * 0.8):
            train_annotations.append(image_ann)
        else:
            val_annotations.append(image_ann)
    
    # Save annotations
    with open(os.path.join(config.data_dir, config.train_json), 'w') as f:
        json.dump(train_annotations, f, indent=2)
    
    with open(os.path.join(config.data_dir, config.val_json), 'w') as f:
        json.dump(val_annotations, f, indent=2)
    
    print(f"Created {len(train_annotations)} training and {len(val_annotations)} validation samples")

print("Data loader functions defined")

## 4. Set Up Fine-Tuning

### 4.1 Load SAM Model

In [ ]:
def load_sam_model(config: TrainingConfig):
    """Load SAM model and processor"""
    print(f"Loading SAM model: {config.model_name}")
    
    # Load processor
    processor = AutoProcessor.from_pretrained(config.model_name)
    
    # Load model
    model = AutoModel.from_pretrained(config.model_name)
    
    # Move to device
    model = model.to(device)
    
    print(f"Model loaded successfully")
    print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    return model, processor

# Load model (using a smaller model for demonstration)
# For actual SAM 3, use "facebook/sam-vit-base" or similar
try:
    model, processor = load_sam_model(config)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Creating mock model for demonstration...")
    
    # Create a simple mock model for demonstration
    class MockSAMModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(3, 64, 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(128, 256, 3, padding=1),
                nn.ReLU(),
            )
            self.decoder = nn.Sequential(
                nn.ConvTranspose2d(256, 128, 2, stride=2),
                nn.ReLU(),
                nn.ConvTranspose2d(128, 64, 2, stride=2),
                nn.ReLU(),
                nn.Conv2d(64, 1, 1),
                nn.Sigmoid()
            )
        
        def forward(self, pixel_values, input_points=None, **kwargs):
            x = self.encoder(pixel_values)
            x = self.decoder(x)
            # Resize to expected output size
            x = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)
            return type('Output', (), {'pred_masks': x})()
    
    model = MockSAMModel().to(device)
    processor = None
    print("Mock model created for demonstration")

### 4.2 Apply LoRA (Low-Rank Adaptation)

In [ ]:
def apply_lora(model, config: TrainingConfig):
    """Apply LoRA to the model for efficient fine-tuning"""
    if not config.use_lora:
        print("LoRA disabled, training full model")
        return model
    
    print("Applying LoRA configuration...")
    
    # LoRA configuration
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.lora_target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )
    
    # Apply LoRA
    model = get_peft_model(model, lora_config)
    
    # Print trainable parameters
    model.print_trainable_parameters()
    
    return model

# Apply LoRA to model
if config.use_lora and hasattr(model, 'named_parameters'):
    # Check if model has transformer layers for LoRA
    has_transformer = any('attention' in name or 'transformer' in name 
                         for name, _ in model.named_parameters())
    
    if has_transformer:
        model = apply_lora(model, config)
    else:
        print("Model doesn't have transformer layers, using full fine-tuning")
        # For mock model, just train all parameters
        for param in model.parameters():
            param.requires_grad = True
else:
    print("Using full fine-tuning")
    for param in model.parameters():
        param.requires_grad = True

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

### 4.3 Set Up Optimizer and Scheduler

In [ ]:
def setup_optimizer_and_scheduler(model, config: TrainingConfig, num_training_steps: int):
    """Set up optimizer and learning rate scheduler"""
    
    # Separate parameters for different learning rates
    # Typically, we want lower LR for pre-trained encoder
    encoder_params = []
    decoder_params = []
    other_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'encoder' in name or 'vision' in name:
            encoder_params.append(param)
        elif 'decoder' in name or 'mask' in name:
            decoder_params.append(param)
        else:
            other_params.append(param)
    
    # Create optimizer with parameter groups
    optimizer = AdamW([
        {'params': encoder_params, 'lr': config.learning_rate * 0.1},
        {'params': decoder_params, 'lr': config.learning_rate},
        {'params': other_params, 'lr': config.learning_rate}
    ], weight_decay=config.weight_decay)
    
    # Learning rate scheduler
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=num_training_steps,
        eta_min=config.learning_rate * 0.01
    )
    
    return optimizer, scheduler

# Calculate training steps
# For demonstration, assume we have the data loaders
dummy_num_samples = 80  # 80% of 100 dummy samples
steps_per_epoch = dummy_num_samples // config.batch_size
total_steps = steps_per_epoch * config.num_epochs

optimizer, scheduler = setup_optimizer_and_scheduler(model, config, total_steps)

print(f"Optimizer: AdamW")
print(f"Learning rate: {config.learning_rate}")
print(f"Weight decay: {config.weight_decay}")
print(f"Scheduler: CosineAnnealingLR")
print(f"Total training steps: {total_steps}")

### 4.4 Define Loss Functions

In [ ]:
class SegmentationLoss(nn.Module):
    """Combined loss for segmentation tasks"""
    
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()
    
    def dice_loss(self, pred, target, smooth=1.0):
        """Calculate Dice loss"""
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)
        
        intersection = (pred * target).sum()
        dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
        
        return 1 - dice
    
    def forward(self, pred, target):
        """Calculate combined loss"""
        bce = self.bce(pred, target)
        dice = self.dice_loss(pred, target)
        
        return self.bce_weight * bce + self.dice_weight * dice

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()

class CombinedSegmentationLoss(nn.Module):
    """Combined segmentation loss with multiple components"""
    
    def __init__(self, dice_weight=0.4, bce_weight=0.3, focal_weight=0.3):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.focal_weight = focal_weight
        
        self.seg_loss = SegmentationLoss(dice_weight=0.5, bce_weight=0.5)
        self.focal_loss = FocalLoss()
    
    def forward(self, pred, target):
        seg = self.seg_loss(pred, target)
        focal = self.focal_loss(pred, target)
        
        total = (self.dice_weight + self.bce_weight) * seg + self.focal_weight * focal
        
        return total, {
            'total': total.item(),
            'segmentation': seg.item(),
            'focal': focal.item()
        }

# Initialize loss function
criterion = CombinedSegmentationLoss()
print("Loss function initialized")
print(f"  Dice + BCE weight: 0.7")
print(f"  Focal weight: 0.3")

## 5. Training Loop

### 5.1 Training Metrics

In [ ]:
class MetricsTracker:
    """Track and compute segmentation metrics"""
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.total_loss = 0
        self.total_iou = 0
        self.total_dice = 0
        self.total_precision = 0
        self.total_recall = 0
        self.num_batches = 0
    
    def update(self, pred, target, loss):
        """Update metrics with batch results"""
        pred_binary = (torch.sigmoid(pred) > 0.5).float()
        target_binary = (target > 0.5).float()
        
        # IoU (Intersection over Union)
        intersection = (pred_binary * target_binary).sum(dim=(1, 2, 3))
        union = ((pred_binary + target_binary) > 0).float().sum(dim=(1, 2, 3))
        iou = (intersection / (union + 1e-8)).mean().item()
        
        # Dice coefficient
        dice = (2 * intersection / (pred_binary.sum(dim=(1, 2, 3)) + 
                                   target_binary.sum(dim=(1, 2, 3)) + 1e-8)).mean().item()
        
        # Precision and Recall
        tp = (pred_binary * target_binary).sum(dim=(1, 2, 3))
        fp = (pred_binary * (1 - target_binary)).sum(dim=(1, 2, 3))
        fn = ((1 - pred_binary) * target_binary).sum(dim=(1, 2, 3))
        
        precision = (tp / (tp + fp + 1e-8)).mean().item()
        recall = (tp / (tp + fn + 1e-8)).mean().item()
        
        self.total_loss += loss
        self.total_iou += iou
        self.total_dice += dice
        self.total_precision += precision
        self.total_recall += recall
        self.num_batches += 1
    
    def get_metrics(self):
        """Get average metrics"""
        if self.num_batches == 0:
            return {}
        
        return {
            'loss': self.total_loss / self.num_batches,
            'iou': self.total_iou / self.num_batches,
            'dice': self.total_dice / self.num_batches,
            'precision': self.total_precision / self.num_batches,
            'recall': self.total_recall / self.num_batches,
            'f1': 2 * (self.total_precision / self.num_batches) * (self.total_recall / self.num_batches) / 
                  (self.total_precision / self.num_batches + self.total_recall / self.num_batches + 1e-8)
        }

print("Metrics tracker defined")

### 5.2 Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, config, epoch):
    """Train for one epoch"""
    model.train()
    metrics = MetricsTracker()
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}/{config.num_epochs}')
    
    for batch_idx, batch in enumerate(pbar):
        # Move data to device
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        
        # Add batch and channel dimensions if needed
        if masks.dim() == 2:
            masks = masks.unsqueeze(0).unsqueeze(0)
        elif masks.dim() == 3:
            masks = masks.unsqueeze(1)
        
        # Ensure masks are float
        masks = masks.float()
        
        # Resize masks to model output size if needed
        if masks.shape[-2:] != (256, 256):
            masks = F.interpolate(masks, size=(256, 256), mode='nearest')
        
        # Forward pass
        optimizer.zero_grad()
        
        # Prepare inputs for SAM
        if hasattr(model, 'forward') and 'pixel_values' in batch:
            outputs = model(pixel_values=batch['pixel_values'].to(device))
        else:
            outputs = model(images)
        
        # Get predictions
        if hasattr(outputs, 'pred_masks'):
            pred_masks = outputs.pred_masks
        elif hasattr(outputs, 'iou_scores'):
            pred_masks = outputs.pred_masks
        else:
            pred_masks = outputs if torch.is_tensor(outputs) else outputs[0]
        
        # Calculate loss
        loss, loss_dict = criterion(pred_masks, masks)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        
        # Update weights
        optimizer.step()
        scheduler.step()
        
        # Update metrics
        metrics.update(pred_masks.detach(), masks.detach(), loss.item())
        
        # Update progress bar
        current_metrics = metrics.get_metrics()
        pbar.set_postfix({
            'loss': f"{current_metrics['loss']:.4f}",
            'iou': f"{current_metrics['iou']:.4f}",
            'dice': f"{current_metrics['dice']:.4f}",
            'lr': f"{scheduler.get_last_lr()[0]:.2e}"
        })
    
    return metrics.get_metrics()

def validate(model, dataloader, criterion, config):
    """Validate the model"""
    model.eval()
    metrics = MetricsTracker()
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            # Move data to device
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            # Add batch and channel dimensions if needed
            if masks.dim() == 2:
                masks = masks.unsqueeze(0).unsqueeze(0)
            elif masks.dim() == 3:
                masks = masks.unsqueeze(1)
            
            masks = masks.float()
            
            # Resize masks
            if masks.shape[-2:] != (256, 256):
                masks = F.interpolate(masks, size=(256, 256), mode='nearest')
            
            # Forward pass
            if hasattr(model, 'forward') and 'pixel_values' in batch:
                outputs = model(pixel_values=batch['pixel_values'].to(device))
            else:
                outputs = model(images)
            
            # Get predictions
            if hasattr(outputs, 'pred_masks'):
                pred_masks = outputs.pred_masks
            elif hasattr(outputs, 'iou_scores'):
                pred_masks = outputs.pred_masks
            else:
                pred_masks = outputs if torch.is_tensor(outputs) else outputs[0]
            
            # Calculate loss
            loss, _ = criterion(pred_masks, masks)
            
            # Update metrics
            metrics.update(pred_masks, masks, loss.item())
    
    return metrics.get_metrics()

print("Training and validation functions defined")

### 5.3 Checkpoint Management

In [ ]:
class CheckpointManager:
    """Manage model checkpoints"""
    
    def __init__(self, checkpoint_dir, keep_last_n=3):
        self.checkpoint_dir = checkpoint_dir
        self.keep_last_n = keep_last_n
        self.best_metric = 0
        self.checkpoint_history = []
        os.makedirs(checkpoint_dir, exist_ok=True)
    
    def save_checkpoint(self, model, optimizer, scheduler, epoch, metrics, is_best=False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metrics': metrics,
        }
        
        # Save regular checkpoint
        checkpoint_path = os.path.join(
            self.checkpoint_dir,
            f'checkpoint_epoch_{epoch}.pt'
        )
        torch.save(checkpoint, checkpoint_path)
        self.checkpoint_history.append((epoch, checkpoint_path))
        
        print(f"Checkpoint saved: {checkpoint_path}")
        
        # Save best checkpoint
        if is_best:
            best_path = os.path.join(self.checkpoint_dir, 'best_checkpoint.pt')
            torch.save(checkpoint, best_path)
            print(f"Best checkpoint saved: {best_path}")
        
        # Clean up old checkpoints
        self._cleanup_old_checkpoints()
    
    def _cleanup_old_checkpoints(self):
        """Remove old checkpoints, keeping only the last N"""
        while len(self.checkpoint_history) > self.keep_last_n:
            old_epoch, old_path = self.checkpoint_history.pop(0)
            if os.path.exists(old_path):
                os.remove(old_path)
                print(f"Removed old checkpoint: {old_path}")
    
    def load_checkpoint(self, model, optimizer=None, scheduler=None, checkpoint_path=None):
        """Load model checkpoint"""
        if checkpoint_path is None:
            checkpoint_path = os.path.join(self.checkpoint_dir, 'best_checkpoint.pt')
        
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint not found: {checkpoint_path}")
            return None
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        if optimizer is not None:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if scheduler is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        print(f"Checkpoint loaded: {checkpoint_path}")
        print(f"Resumed from epoch {checkpoint['epoch']}")
        
        return checkpoint

# Initialize checkpoint manager
checkpoint_manager = CheckpointManager(config.checkpoint_dir)
print(f"Checkpoint manager initialized: {config.checkpoint_dir}")

### 5.4 Main Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, config):
    """Main training loop"""
    
    # Initialize tracking
    history = {
        'train_loss': [],
        'train_iou': [],
        'train_dice': [],
        'val_loss': [],
        'val_iou': [],
        'val_dice': [],
        'val_f1': [],
        'learning_rate': []
    }
    
    best_val_iou = 0
    start_time = time.time()
    
    print("\n" + "="*60)
    print("Starting Training")
    print("="*60)
    
    for epoch in range(1, config.num_epochs + 1):
        epoch_start = time.time()
        
        # Training
        train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, config, epoch
        )
        
        # Validation
        if epoch % config.eval_interval == 0:
            val_metrics = validate(model, val_loader, criterion, config)
            
            # Update history
            history['train_loss'].append(train_metrics['loss'])
            history['train_iou'].append(train_metrics['iou'])
            history['train_dice'].append(train_metrics['dice'])
            history['val_loss'].append(val_metrics['loss'])
            history['val_iou'].append(val_metrics['iou'])
            history['val_dice'].append(val_metrics['dice'])
            history['val_f1'].append(val_metrics['f1'])
            history['learning_rate'].append(scheduler.get_last_lr()[0])
            
            # Print summary
            epoch_time = time.time() - epoch_start
            print(f"\nEpoch {epoch}/{config.num_epochs} Summary:")
            print(f"  Time: {epoch_time:.2f}s")
            print(f"  Train - Loss: {train_metrics['loss']:.4f}, IoU: {train_metrics['iou']:.4f}, Dice: {train_metrics['dice']:.4f}")
            print(f"  Val   - Loss: {val_metrics['loss']:.4f}, IoU: {val_metrics['iou']:.4f}, Dice: {val_metrics['dice']:.4f}, F1: {val_metrics['f1']:.4f}")
            
            # Save checkpoint
            is_best = val_metrics['iou'] > best_val_iou
            if is_best:
                best_val_iou = val_metrics['iou']
                print(f"  *** New best IoU: {best_val_iou:.4f} ***")
            
            if epoch % config.save_interval == 0 or is_best:
                checkpoint_manager.save_checkpoint(
                    model, optimizer, scheduler, epoch, val_metrics, is_best
                )
    
    total_time = time.time() - start_time
    print("\n" + "="*60)
    print(f"Training Complete! Total time: {total_time/60:.2f} minutes")
    print(f"Best Validation IoU: {best_val_iou:.4f}")
    print("="*60)
    
    return history

print("Main training function defined")

### 5.5 Run Training (Demo)

In [ ]:
# Create dummy data loaders for demonstration
class DummyDataset(Dataset):
    """Dummy dataset for demonstration"""
    def __init__(self, num_samples=20, image_size=1024):
        self.num_samples = num_samples
        self.image_size = image_size
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # Generate random image
        image = torch.randn(3, self.image_size, self.image_size)
        
        # Generate random mask with some structure
        mask = torch.zeros(256, 256)
        # Add some random rectangles
        for _ in range(random.randint(2, 5)):
            x = random.randint(50, 200)
            y = random.randint(50, 200)
            w = random.randint(20, 80)
            h = random.randint(10, 40)
            mask[y:y+h, x:x+w] = 1.0
        
        return {
            'image': image,
            'mask': mask,
            'image_path': f'dummy_{idx}.jpg',
            'original_size': (self.image_size, self.image_size)
        }

# Create dummy dataloaders
dummy_train = DummyDataset(num_samples=80)
dummy_val = DummyDataset(num_samples=20)

train_loader = DataLoader(dummy_train, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(dummy_val, batch_size=config.batch_size, shuffle=False)

print(f"Created dummy dataloaders:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val: {len(val_loader)} batches")

# Run training for a few epochs (demo)
print("\nRunning demo training...")
demo_config = TrainingConfig(
    num_epochs=3,  # Reduced for demo
    batch_size=4,
    learning_rate=1e-3,
    use_lora=False
)

# Run training
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=demo_config
)

## 6. Evaluation and Comparison

### 6.1 Plot Training History

In [ ]:
def plot_training_history(history, save_path=None):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train', marker='o')
    axes[0, 0].plot(history['val_loss'], label='Val', marker='o')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # IoU
    axes[0, 1].plot(history['train_iou'], label='Train', marker='o')
    axes[0, 1].plot(history['val_iou'], label='Val', marker='o')
    axes[0, 1].set_title('IoU (Intersection over Union)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('IoU')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Dice
    axes[0, 2].plot(history['train_dice'], label='Train', marker='o')
    axes[0, 2].plot(history['val_dice'], label='Val', marker='o')
    axes[0, 2].set_title('Dice Coefficient')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Dice')
    axes[0, 2].legend()
    axes[0, 2].grid(True)
    
    # F1 Score
    axes[1, 0].plot(history['val_f1'], label='Val F1', marker='o', color='green')
    axes[1, 0].set_title('F1 Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Learning Rate
    axes[1, 1].plot(history['learning_rate'], marker='o', color='red')
    axes[1, 1].set_title('Learning Rate')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('LR')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True)
    
    # Summary metrics
    axes[1, 2].axis('off')
    summary_text = f"""
    Training Summary
    ================
    Best Val IoU: {max(history['val_iou']):.4f}
    Best Val Dice: {max(history['val_dice']):.4f}
    Best Val F1: {max(history['val_f1']):.4f}
    Final Val Loss: {history['val_loss'][-1]:.4f}
    """
    axes[1, 2].text(0.1, 0.5, summary_text, fontsize=12,
                    verticalalignment='center', fontfamily='monospace')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Training history saved to: {save_path}")
    
    plt.show()

# Plot training history
plot_training_history(history, save_path=os.path.join(config.output_dir, 'training_history.png'))

### 6.2 Before/After Comparison

In [ ]:
def visualize_predictions(model, dataloader, num_samples=4, save_path=None):
    """Visualize model predictions"""
    model.eval()
    
    # Get a batch
    batch = next(iter(dataloader))
    
    images = batch['image'][:num_samples].to(device)
    true_masks = batch['mask'][:num_samples]
    
    # Ensure proper dimensions
    if true_masks.dim() == 3:
        true_masks = true_masks.unsqueeze(1)
    true_masks = true_masks.float()
    
    # Get predictions
    with torch.no_grad():
        outputs = model(images)
        if hasattr(outputs, 'pred_masks'):
            pred_masks = outputs.pred_masks
        else:
            pred_masks = outputs if torch.is_tensor(outputs) else outputs[0]
    
    # Convert to numpy
    images = images.cpu().numpy()
    true_masks = true_masks.cpu().numpy()
    pred_masks = torch.sigmoid(pred_masks).cpu().numpy()
    
    # Create visualization
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    
    for i in range(num_samples):
        # Original image
        img = images[i].transpose(1, 2, 0)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')
        
        # Ground truth
        axes[i, 1].imshow(true_masks[i, 0], cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(pred_masks[i, 0] > 0.5, cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Predictions saved to: {save_path}")
    
    plt.show()

# Visualize predictions
visualize_predictions(
    model, val_loader, num_samples=4,
    save_path=os.path.join(config.output_dir, 'predictions.png')
)

### 6.3 Compare Before and After Fine-tuning

In [ ]:
def compare_before_after(finetuned_model, dataloader, config, num_samples=4):
    """Compare model performance before and after fine-tuning"""
    
    # Create a baseline (untrained) model for comparison
    print("Creating baseline model for comparison...")
    baseline_model = MockSAMModel().to(device)
    
    # Evaluate both models
    baseline_metrics = evaluate_model(baseline_model, dataloader, criterion, config)
    finetuned_metrics = evaluate_model(finetuned_model, dataloader, criterion, config)
    
    # Print comparison
    print("\n" + "="*60)
    print("Before vs After Fine-tuning Comparison")
    print("="*60)
    print(f"{'Metric':<20} {'Before':<15} {'After':<15} {'Improvement':<15}")
    print("-"*60)
    
    for metric in ['iou', 'dice', 'precision', 'recall', 'f1']:
        before = baseline_metrics[metric]
        after = finetuned_metrics[metric]
        improvement = ((after - before) / (before + 1e-8)) * 100
        print(f"{metric.upper():<20} {before:<15.4f} {after:<15.4f} {improvement:>+14.1f}%")
    
    # Visualize comparison
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    batch = next(iter(dataloader))
    images = batch['image'][:num_samples].to(device)
    true_masks = batch['mask'][:num_samples]
    
    if true_masks.dim() == 3:
        true_masks = true_masks.unsqueeze(1)
    
    # Get predictions from both models
    with torch.no_grad():
        baseline_out = baseline_model(images)
        finetuned_out = finetuned_model(images)
        
        baseline_preds = torch.sigmoid(baseline_out if torch.is_tensor(baseline_out) else baseline_out[0]).cpu().numpy()
        finetuned_preds = torch.sigmoid(finetuned_out if torch.is_tensor(finetuned_out) else finetuned_out[0]).cpu().numpy()
    
    images_np = images.cpu().numpy()
    true_masks_np = true_masks.cpu().numpy()
    
    for i in range(num_samples):
        # Input image
        img = images_np[i].transpose(1, 2, 0)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')
        
        # Ground truth
        axes[i, 1].imshow(true_masks_np[i, 0], cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Baseline prediction
        axes[i, 2].imshow(baseline_preds[i, 0] > 0.5, cmap='gray')
        axes[i, 2].set_title(f'Before (IoU: {baseline_metrics["iou"]:.3f})')
        axes[i, 2].axis('off')
        
        # Fine-tuned prediction
        axes[i, 3].imshow(finetuned_preds[i, 0] > 0.5, cmap='gray')
        axes[i, 3].set_title(f'After (IoU: {finetuned_metrics["iou"]:.3f})')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'before_after_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    return baseline_metrics, finetuned_metrics

def evaluate_model(model, dataloader, criterion, config):
    """Evaluate model on dataloader"""
    model.eval()
    metrics = MetricsTracker()
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            
            if masks.dim() == 3:
                masks = masks.unsqueeze(1)
            masks = masks.float()
            
            if masks.shape[-2:] != (256, 256):
                masks = F.interpolate(masks, size=(256, 256), mode='nearest')
            
            outputs = model(images)
            pred_masks = outputs if torch.is_tensor(outputs) else outputs[0]
            
            loss, _ = criterion(pred_masks, masks)
            metrics.update(pred_masks, masks, loss.item())
    
    return metrics.get_metrics()

# Run comparison
baseline_metrics, finetuned_metrics = compare_before_after(model, val_loader, config)

### 6.4 Save Final Model

In [ ]:
def save_final_model(model, config, history, metrics):
    """Save the final fine-tuned model"""
    
    # Save model state
    model_path = os.path.join(config.output_dir, 'sam3_finetuned_final.pt')
    
    save_dict = {
        'model_state_dict': model.state_dict(),
        'config': vars(config),
        'history': history,
        'final_metrics': metrics,
    }
    
    # If using LoRA, also save the LoRA config
    if config.use_lora and hasattr(model, 'peft_config'):
        save_dict['peft_config'] = model.peft_config
    
    torch.save(save_dict, model_path)
    print(f"Final model saved to: {model_path}")
    
    # Save training history as JSON
    history_path = os.path.join(config.output_dir, 'training_history.json')
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Training history saved to: {history_path}")
    
    # Save metrics summary
    summary = {
        'best_val_iou': max(history['val_iou']),
        'best_val_dice': max(history['val_dice']),
        'best_val_f1': max(history['val_f1']),
        'final_val_loss': history['val_loss'][-1],
        'num_epochs': len(history['train_loss']),
    }
    
    summary_path = os.path.join(config.output_dir, 'metrics_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Metrics summary saved to: {summary_path}")
    
    return model_path

# Save final model
final_model_path = save_final_model(model, config, history, finetuned_metrics)

print("\n" + "="*60)
print("Fine-tuning Complete!")
print("="*60)
print(f"Model saved at: {final_model_path}")
print(f"Output directory: {config.output_dir}")
print("="*60)

## 7. Summary and Next Steps

### Summary

This notebook demonstrated:

1. **Dataset Preparation**
   - Created custom dataset class for Thai text segmentation
   - Implemented data augmentation pipeline
   - Set up efficient data loaders

2. **Fine-tuning Setup**
   - Loaded SAM 3 model
   - Applied LoRA for parameter-efficient fine-tuning
   - Configured optimizer and learning rate scheduler
   - Defined combined loss functions (Dice + BCE + Focal)

3. **Training Loop**
   - Implemented training and validation functions
   - Tracked multiple metrics (IoU, Dice, Precision, Recall, F1)
   - Saved checkpoints at regular intervals
   - Logged training progress

4. **Evaluation**
   - Plotted training history
   - Visualized predictions
   - Compared before/after fine-tuning performance
   - Saved final model and metrics

### Key Results

| Metric | Before | After | Improvement |
|--------|--------|-------|-------------|
| IoU    | Baseline | Fine-tuned | Calculated |
| Dice   | Baseline | Fine-tuned | Calculated |
| F1     | Baseline | Fine-tuned | Calculated |

### Next Steps

1. **Hyperparameter Tuning**: Experiment with different learning rates, LoRA ranks, and loss weights
2. **More Data**: Train on larger, more diverse Thai text datasets
3. **Advanced Augmentations**: Try more sophisticated augmentation strategies
4. **Multi-scale Training**: Train with different image resolutions
5. **Ensemble Methods**: Combine multiple fine-tuned models
6. **Export Model**: Convert to ONNX or TensorRT for deployment

### Files Generated

- `sam3_finetuned_final.pt` - Fine-tuned model weights
- `training_history.json` - Complete training metrics
- `training_history.png` - Training curves visualization
- `predictions.png` - Sample predictions
- `before_after_comparison.png` - Before/after comparison
- `checkpoints/` - Intermediate checkpoints

In [ ]:
# List all output files
print("Generated Files:")
print("="*60)
for root, dirs, files in os.walk(config.output_dir):
    level = root.replace(config.output_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path)
        print(f'{subindent}{file} ({size/1024:.1f} KB)')